# Système de recommandation agricole - Modélisation

## Librairies nécessaires

In [11]:
# Import de base
import pandas as pd
import numpy as np
import sys,os
sys.path.append(os.path.abspath(".."))

# Import scikit learn
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor

# Import des autres modèles testés
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Import mlflow
import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("file://" + os.path.abspath("../mlruns"))
from mlflow.tracking import MlflowClient

# Import du projet
from scripts.preprocessing_pipeline import (
    separation_X_y,
    preparation_pipeline,
    cross_validation,
    train_predict
)

### Comparaison modèle sans optimisation 

In [12]:
def mlflow_tracking_model(model,model_name,tags,projet_description):
    # ==========================================
    # Configuration MLflow
    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.set_experiment("Yield_Prediction_Agricultural")

    # L'autolog va capturer automatiquement les paramètres du modèle
    mlflow.sklearn.autolog(log_models=True, log_datasets=False, silent=True)

    # ==========================================
    # Chargement des données
    df = pd.read_csv("../data/processed/yield_df_final_conso_encoded.csv", index_col=0)
    model_name
    tags
    projet_description


    with mlflow.start_run(run_name=model_name, tags={"Training Info" : tags,
                            "mlflow.note.content" : projet_description
                            }):

        # Logs infos générales (Metadata)
        mlflow.log_params({
            "dataset": "yield_df_final_conso_encoded.csv",
            "n_rows": df.shape[0],
            "n_cols": df.shape[1],
            "random_state": 42
        })

        # Separation
        X_train, X_test, y_train, y_test, categorical_cols, numeric_cols = separation_X_y(df)

        # ==========================================
        # Modèle et Pipeline
        model = model
        pipeline = preparation_pipeline(
            numeric_cols=numeric_cols,
            categorical_cols=categorical_cols,
            model=model
        )

        # ==========================================
        # Cross-validation
        cv_results = cross_validation(
            pipeline=pipeline,
            X_train=X_train,
            y_train=y_train
        )
        
        # Calcul des scores CV
        cv_metrics = {
            "cv_rmse": np.sqrt(-cv_results["test_mse"]).mean(),
            "cv_mae": (-cv_results["test_mae"]).mean(),
            "cv_mape": (-cv_results["test_mape"]).mean(),
            "cv_r2": cv_results["test_r2"].mean()
        }
        mlflow.log_metrics(cv_metrics)

        # ==========================================
        # Entraînement Final (L'Autolog capture tout ici)
        pipeline.fit(X_train, y_train)

        # ==========================================
        # Prédictions et Test Metrics
        y_pred = pipeline.predict(X_test)
       # ==========================================
        # Erreur relative moyenne (%)
        # epsilon pour éviter division par zéro
        epsilon = 1e-6
        relative_errors = np.abs((y_test - y_pred) / (np.abs(y_test) + epsilon))
        mape_pct = np.mean(relative_errors) * 100

        # Variante plus "métier" : précision relative en %
        precision_relative_pct = max(0, 100 - mape_pct)
        # taux de décisions jugées exploitables (erreur <= 10%)
        profitable_decision_rate_pct = (relative_errors <= 0.10).mean() * 100

        test_metrics = {
            "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
            "test_mae": mean_absolute_error(y_test, y_pred),
            "test_mape": mean_absolute_percentage_error(y_test, y_pred),
            "test_r2": r2_score(y_test, y_pred),
            "profitability_proxy_pct": precision_relative_pct,
            "profitable_decision_rate_pct": profitable_decision_rate_pct
        }

        mlflow.log_metrics(test_metrics)

        # mlflow.sklearn.log_model(pipeline, "model", pip_requirements=["scikit-learn", "pandas", "numpy"])

        print("\n=== Modèle avec Autolog & CV ===")
        print(f"CV RMSE   : {cv_metrics['cv_rmse']:.4f}")
        print(f"CV MAE   : {cv_metrics['cv_mae']:.4f}")
        print(f"CV MAPE   : {cv_metrics['cv_mape']:.4f}")
        print(f"CV R2   : {cv_metrics['cv_r2']:.4f}")
        print(f"Test RMSE : {test_metrics['test_rmse']:.4f}")
        print(f"Test R2   : {test_metrics['test_r2']:.4f}")
        print(f"Test MAE   : {test_metrics['test_mae']:.4f}")
        print(f"Test MAPE   : {test_metrics['test_mape']:.4f}")
        print(f"Profitability_proxy_pct   : {test_metrics['profitability_proxy_pct']:.4f}")
        print(f"Profitable_decision_rate_pct   : {test_metrics['profitable_decision_rate_pct']:.4f}")

In [14]:
# DummyRegressor
model = DummyRegressor()
model_name = "DummyRegressor - Baseline"
tags = "DummyRegressor - Baseline"
projet_description = "Test d'un modèle DummyRegressor pour avoir une base sur laquelle comparer"
mlflow_tracking_model(model, model_name, tags, projet_description)

2026-03-31 16:11:42,983 - INFO - ['avg_temp', 'rainfall_mm', 'pesticides_tonnes', 'tech_trend', 'climate_instability', 'relative_tech_intensity']
2026-03-31 16:11:42,984 - INFO - ['region_australia_and_new_zealand', 'region_central_asia', 'region_eastern_asia', 'region_eastern_europe', 'region_latin_america_and_the_caribbean', 'region_melanesia', 'region_micronesia', 'region_northern_africa', 'region_northern_america', 'region_northern_europe', 'region_polynesia', 'region_south-eastern_asia', 'region_southern_asia', 'region_southern_europe', 'region_sub-saharan_africa', 'region_western_asia', 'region_western_europe', 'item_cassava', 'item_maize', 'item_plantains_and_others', 'item_potatoes', 'item_rice', 'item_sorghum', 'item_soybean', 'item_sweet_potatoes', 'item_wheat', 'item_yams']



=== Modèle avec Autolog & CV ===
CV RMSE   : 75652.6639
CV MAE   : 55591.4405
CV MAPE   : 107888103053965168.0000
CV R2   : -0.0007
Test RMSE : 73864.1004
Test R2   : -0.0012
Test MAE   : 54567.1261
Test MAPE   : 2.8247
Profitability_proxy_pct   : 0.0000
Profitable_decision_rate_pct   : 5.3164
🏃 View run DummyRegressor - Baseline at: http://127.0.0.1:5000/#/experiments/1/runs/cede47a0bffb48dd86c625d81638ad77
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [15]:
# Modèle de régression linéaire
model = LinearRegression()
model_name = "LinearRegression - Baseline"
tags = "LinearRegression - Baseline"
projet_description = "Test d'un modèle LinearRegression sans optimisation"
mlflow_tracking_model(model, model_name, tags, projet_description)


2026-03-31 16:11:50,650 - INFO - ['avg_temp', 'rainfall_mm', 'pesticides_tonnes', 'tech_trend', 'climate_instability', 'relative_tech_intensity']
2026-03-31 16:11:50,651 - INFO - ['region_australia_and_new_zealand', 'region_central_asia', 'region_eastern_asia', 'region_eastern_europe', 'region_latin_america_and_the_caribbean', 'region_melanesia', 'region_micronesia', 'region_northern_africa', 'region_northern_america', 'region_northern_europe', 'region_polynesia', 'region_south-eastern_asia', 'region_southern_asia', 'region_southern_europe', 'region_sub-saharan_africa', 'region_western_asia', 'region_western_europe', 'item_cassava', 'item_maize', 'item_plantains_and_others', 'item_potatoes', 'item_rice', 'item_sorghum', 'item_soybean', 'item_sweet_potatoes', 'item_wheat', 'item_yams']



=== Modèle avec Autolog & CV ===
CV RMSE   : 48452.9898
CV MAE   : 32395.6681
CV MAPE   : 14857077228332300.0000
CV R2   : 0.5895
Test RMSE : 48180.7848
Test R2   : 0.5740
Test MAE   : 32150.9483
Test MAPE   : 1.0347
Profitability_proxy_pct   : 0.0000
Profitable_decision_rate_pct   : 12.8108
🏃 View run LinearRegression - Baseline at: http://127.0.0.1:5000/#/experiments/1/runs/7f134540c87b4211b8ce47764174fe5d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [16]:
# Modèle de RandomForest
model = RandomForestRegressor(random_state=42)
model_name = "RandomForest - Baseline"
tags = "RandomForest - Baseline"
projet_description = "Test d'un modèle Random Forest sans optimisation"
mlflow_tracking_model(model, model_name, tags, projet_description)


2026-03-31 16:12:04,124 - INFO - ['avg_temp', 'rainfall_mm', 'pesticides_tonnes', 'tech_trend', 'climate_instability', 'relative_tech_intensity']
2026-03-31 16:12:04,124 - INFO - ['region_australia_and_new_zealand', 'region_central_asia', 'region_eastern_asia', 'region_eastern_europe', 'region_latin_america_and_the_caribbean', 'region_melanesia', 'region_micronesia', 'region_northern_africa', 'region_northern_america', 'region_northern_europe', 'region_polynesia', 'region_south-eastern_asia', 'region_southern_asia', 'region_southern_europe', 'region_sub-saharan_africa', 'region_western_asia', 'region_western_europe', 'item_cassava', 'item_maize', 'item_plantains_and_others', 'item_potatoes', 'item_rice', 'item_sorghum', 'item_soybean', 'item_sweet_potatoes', 'item_wheat', 'item_yams']



=== Modèle avec Autolog & CV ===
CV RMSE   : 18127.0200
CV MAE   : 8505.0325
CV MAPE   : 25836986908721824.0000
CV R2   : 0.9423
Test RMSE : 17486.8506
Test R2   : 0.9439
Test MAE   : 7796.2916
Test MAPE   : 0.1865
Profitability_proxy_pct   : 81.3503
Profitable_decision_rate_pct   : 60.4699
🏃 View run RandomForest - Baseline at: http://127.0.0.1:5000/#/experiments/1/runs/050eeceb2b65448ea90d1e5772a7699c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [17]:
# Modèle de XGBoost
model = xgb.XGBRegressor(random_state=42)
model_name = "XGBRegressor - Baseline"
tags = "XGBRegressor - Baseline"
projet_description = "Test d'un modèle XGBRegressor sans optimisation"
mlflow_tracking_model(model, model_name, tags, projet_description)


2026-03-31 16:12:22,972 - INFO - ['avg_temp', 'rainfall_mm', 'pesticides_tonnes', 'tech_trend', 'climate_instability', 'relative_tech_intensity']
2026-03-31 16:12:22,972 - INFO - ['region_australia_and_new_zealand', 'region_central_asia', 'region_eastern_asia', 'region_eastern_europe', 'region_latin_america_and_the_caribbean', 'region_melanesia', 'region_micronesia', 'region_northern_africa', 'region_northern_america', 'region_northern_europe', 'region_polynesia', 'region_south-eastern_asia', 'region_southern_asia', 'region_southern_europe', 'region_sub-saharan_africa', 'region_western_asia', 'region_western_europe', 'item_cassava', 'item_maize', 'item_plantains_and_others', 'item_potatoes', 'item_rice', 'item_sorghum', 'item_soybean', 'item_sweet_potatoes', 'item_wheat', 'item_yams']



=== Modèle avec Autolog & CV ===
CV RMSE   : 21165.9514
CV MAE   : 12710.3711
CV MAPE   : 17832065551355084.0000
CV R2   : 0.9216
Test RMSE : 20362.0486
Test R2   : 0.9239
Test MAE   : 12160.6406
Test MAPE   : 0.3999
Profitability_proxy_pct   : 60.0106
Profitable_decision_rate_pct   : 31.7098
🏃 View run XGBRegressor - Baseline at: http://127.0.0.1:5000/#/experiments/1/runs/a6652b1c855c41debe3fae93cbb31d7f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [18]:
# Modèle de LightGBM
model = lgb.LGBMRegressor(random_state=42, verbose=-1)
model_name = "LGBMRegressor - Baseline"
tags = "LGBMRegressor - Baseline"
projet_description = "Test d'un modèle LGBMRegressor sans optimisation"
mlflow_tracking_model(model, model_name, tags, projet_description)


2026-03-31 16:12:34,067 - INFO - ['avg_temp', 'rainfall_mm', 'pesticides_tonnes', 'tech_trend', 'climate_instability', 'relative_tech_intensity']
2026-03-31 16:12:34,068 - INFO - ['region_australia_and_new_zealand', 'region_central_asia', 'region_eastern_asia', 'region_eastern_europe', 'region_latin_america_and_the_caribbean', 'region_melanesia', 'region_micronesia', 'region_northern_africa', 'region_northern_america', 'region_northern_europe', 'region_polynesia', 'region_south-eastern_asia', 'region_southern_asia', 'region_southern_europe', 'region_sub-saharan_africa', 'region_western_asia', 'region_western_europe', 'item_cassava', 'item_maize', 'item_plantains_and_others', 'item_potatoes', 'item_rice', 'item_sorghum', 'item_soybean', 'item_sweet_potatoes', 'item_wheat', 'item_yams']



=== Modèle avec Autolog & CV ===
CV RMSE   : 26298.6540
CV MAE   : 16708.1375
CV MAPE   : 24803177163291184.0000
CV R2   : 0.8790
Test RMSE : 26035.4246
Test R2   : 0.8756
Test MAE   : 16329.4829
Test MAPE   : 0.5635
Profitability_proxy_pct   : 43.6471
Profitable_decision_rate_pct   : 24.0268
🏃 View run LGBMRegressor - Baseline at: http://127.0.0.1:5000/#/experiments/1/runs/b4bced0212294bd28bcbd3a1268daf49
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


- Sur ces premiers tests, le modèle qui s'en sort le mieux sur l'ensemble des métriques (performance et métier) c'est celui de Random Forest.
- Les résultats de XGBoost et LightGBM sont exploitables pour essayer d'améliorer leur performance.
- Par contre le modèle de régression linéaire est trop loin en terme de performance. Au vu de la distribution de notre varibale cible, cela n'est pas étonnant.